#### 02 — AI-Assisted Customer Feedback Analysis

##### Objective

The previous analysis showed that simple word frequency and TF-IDF can describe customer language but are limited in identifying deeper patterns across complaint narratives.

In this notebook, I will test whether AI-based text embeddings can group complaints according to their meaning rather than relying only on exact words.

The goal is to discover recurring customer themes that can later be evaluated against the existing CFPB issue and sub-issue classifications.

##### 1. AI Approach

I will use text embeddings to represent each customer narrative as a numerical vector.

Complaints with similar meanings should have more similar representations even when the exact wording is different.

These representations will later be used to identify groups of similar customer experiences.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

In [ ]:
## Load the dataset 

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
raw_file = RAW_DATA_DIR / "cfpb_credit_card_complaints.csv"

df = pd.read_csv(raw_file)

print("Dataset shape:", df.shape)

Dataset shape: (10000, 17)


In [ ]:
# Recreate the usable narrative dataset

attachment_only = (
    df["complaint_what_happened"]
    .fillna("")
    .str.strip()
    .str.lower()
    .isin([
        "see the attached documents.",
        "see the attached documents"
    ])
)

text_df = df.loc[~attachment_only].copy()

print("Narratives available for AI analysis:", len(text_df))

Narratives available for AI analysis: 9995


In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'Server disconnected without sending a response.' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [ ]:
## Generate Complaint Embeddings (test)

sample_texts = text_df["complaint_what_happened"].head(5).tolist()

sample_embeddings = model.encode(
    sample_texts,
    show_progress_bar=False
)

print("Embedding shape:", sample_embeddings.shape)

Embedding shape: (5, 384)


##### 3. Generate Complaint Embeddings

The test embedding worked successfully, so I will now generate embeddings for all 9,995 usable complaint narratives.

These embeddings will be used to compare complaints based on their semantic similarity.

In [ ]:
embeddings = model.encode(
    text_df["complaint_what_happened"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embeddings shape: (9995, 384)


#### 4. Finding Similar Customer Complaints

The embeddings allow complaints to be compared based on their semantic similarity.

I will select one complaint and find other complaints with the most similar embeddings. This provides a simple way to check whether the model is grouping customer experiences that appear related.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    embeddings[0].reshape(1, -1),
    embeddings
)[0]

similar_indices = similarity_scores.argsort()[::-1][1:6]

similar_complaints = text_df.iloc[similar_indices][
    ["complaint_id", "issue", "sub_issue", "complaint_what_happened"]
].copy()

similar_complaints["similarity"] = similarity_scores[similar_indices]

similar_complaints

,complaint_id,issue,sub_issue,complaint_what_happened,similarity
2361,9734203,Problem when making payments,Problem during payment process,"I received the response from the CFPB regarding Ally Bank and the response stated Ally purchased the account back from Portfolio Recovery, my account was notated and the representatives at Ally bank would know how to handle my account when I called into set up my payments. \n\nThe harassment has...",0.690842
9255,9043127,Getting a credit card,Sent card you never applied for,"Truest bank called to verify my application for a new credit card along with a cosigner. Cosigner was XXXX XXXX in XXXXXXXX XXXX I do not know this XXXX XXXX, and ask Truitt to cancel the application due to fraud. Truist cancels the card application due to fraud with a cancellation code of XXXX....",0.683815
7349,9196261,Fees or interest,Problem with fees,The bank failed to help. Repeatedly never returned my calls. This has gone on since XXXX of XXXX I called and sent registered mail XXXX and XXXX time a month for XXXX months. \nI paid off all the fraud charges and all that has remained is service charge and interest which I will not pay for thei...,0.678303
9050,9061728,"Other features, terms, or problems",Other problem,"I was contacted today by phone and told that someone used my SSN to apply for a Bank of America credit card. I was surprised to find my XXXX in the public domain. \n\nThe caller identified himself a being with BOA XXXX XXXX, XXXX XXXX. He asked, and I agreed he could pass this to CFPB. \n\nThe c...",0.666214
2775,9689033,"Other features, terms, or problems",Other problem,"I was called on the phone by a company claiming to be my bank and then transferred to a company claiming to be the Consumer Financial Protection Bureau. XXXX XXXX, XXXX. \nXXXX. They claimed to know my credit card debt and offered a restructuring of my debt for a fee of {$4500.00}. This seemed t...",0.655954


In [ ]:
print("Original complaint:")
print(text_df.iloc[0]["complaint_what_happened"])

Original complaint:
Chae bank called me about a fraudulent case application. They gave me a case number XXXX, tranferred me CFPB. During the call, long silent when agent said is looking at the report, then cut off. No call back was done.


##### Finding — Semantic Similarity Can Cross Existing Categories

The embedding model identified complaints with similar language or meaning even when their CFPB issue and sub-issue classifications were different.

This suggests that semantic analysis may reveal relationships between customer experiences that are not captured by the existing taxonomy.

This is an initial observation from a single complaint and will need to be tested across the wider dataset before being treated as a product insight.

#### 5. Category Consistency of Similar Complaints

The initial similarity example showed that semantically similar complaints can belong to different CFPB categories.

I will test this across a sample of complaints to see how often the most similar complaints share the same issue category.

In [ ]:
sample_indices = np.random.RandomState(42).choice(
    len(text_df),
    size=100,
    replace=False
)

same_issue_rates = []

for idx in sample_indices:
    scores = cosine_similarity(
        embeddings[idx].reshape(1, -1),
        embeddings
    )[0]

    nearest_indices = scores.argsort()[::-1][1:6]

    original_issue = text_df.iloc[idx]["issue"]

    same_issue = (
        text_df.iloc[nearest_indices]["issue"] == original_issue
    ).mean()

    same_issue_rates.append(same_issue)

print(
    "Average proportion of top 5 similar complaints "
    "sharing the same issue:",
    round(np.mean(same_issue_rates) * 100, 2),
    "%"
)

Average proportion of top 5 similar complaints sharing the same issue: 47.0 %


##### Finding — Embeddings Provide a Different View of the Taxonomy

Across a sample of 100 complaints, the five nearest semantic neighbors shared the same CFPB issue about 47% of the time on average.

This indicates that semantic similarity does not simply reproduce the existing CFPB issue categories. Many similar complaints are associated with different issue categories.

This suggests that embeddings may provide a useful additional perspective for discovering customer themes, although the semantic matches still need to be evaluated for relevance before being used for product decisions.

##### 6. Customer Theme Clustering

The similarity analysis showed that embeddings can connect complaints across the existing CFPB categories.

I will now group the complaint embeddings into clusters to identify broader patterns in customer experiences.

The clusters are exploratory and will not automatically be treated as final customer problems.

In [ ]:
from sklearn.cluster import KMeans

k = 10

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

text_df["cluster"] = kmeans.fit_predict(embeddings)

print("Clusters created:", text_df["cluster"].nunique())

Clusters created: 10


##### Finding — Initial Customer Clusters

The embedding representations were grouped into 10 exploratory clusters.

The clusters are not being treated as final customer themes yet. K-Means requires the number of clusters to be specified in advance, so the next step is to inspect the complaints within each cluster and determine whether the groups represent meaningful and coherent customer experiences.

##### 7. Inspecting Customer Clusters

I will examine the size and composition of each cluster before deciding whether the clustering produces useful customer themes.

In [ ]:
cluster_summary = []

for cluster in sorted(text_df["cluster"].unique()):
    cluster_data = text_df[text_df["cluster"] == cluster]
    
    cluster_summary.append({
        "cluster": cluster,
        "complaints": len(cluster_data),
        "top_issue": cluster_data["issue"].value_counts().index[0],
        "top_issue_count": cluster_data["issue"].value_counts().iloc[0]
    })

cluster_summary = pd.DataFrame(cluster_summary)

cluster_summary

,cluster,complaints,top_issue,top_issue_count
0,0,793,Getting a credit card,205
1,1,1222,Problem with a purchase shown on your statement,948
2,2,522,Problem with a purchase shown on your statement,173
3,3,785,Problem with a company's investigation into an existing problem,555
4,4,670,Getting a credit card,298
5,5,786,"Advertising and marketing, including promotional offers",266
6,6,1473,Problem when making payments,287
7,7,1253,Getting a credit card,432
8,8,1243,Fees or interest,805
9,9,1248,Problem with a purchase shown on your statement,825


In [ ]:
for cluster in sorted(text_df["cluster"].unique()):
    print("\n" + "=" * 90)
    print("CLUSTER", cluster)
    
    cluster_data = text_df[text_df["cluster"] == cluster]
    
    print("Complaints:", len(cluster_data))
    print("Top issues:")
    print(cluster_data["issue"].value_counts().head(3))
    
    print("\nExample narratives:")
    
    for text in cluster_data["complaint_what_happened"].sample(
        min(3, len(cluster_data)),
        random_state=42
    ):
        print("-", text[:500].replace("\n", " "))


CLUSTER 0
Complaints: 793
Top issues:
issue
Getting a credit card                   205
Incorrect information on your report    125
Other features, terms, or problems       94
Name: count, dtype: int64

Example narratives:
- I, XXXX XXXX, as a federally protected consumer and per the Fair Credit Reporting Act ( 15 USC 1681 ) I am now opting out of any and all authorizations that I may have given you written, unwritten, oral, verbal or nonverbal per U.S.C 6802 effective immediately and indefinitely. XXXX XXXX XXXX
- To Whom It May Concern, I have filed a case with the XXXX XXXX XXXX. On XX/XX/year>, a certificate of service was served to Dollar Bank Federal Savings Bank concerning my Acceptance for Value. This includes a " Request Regarding a Statement of Account '' for account numbers listed in the Affidavit of Truth.   On XX/XX/year>, I sent a proof of claim and XXXX XXXX XXXX XXXX XXXXXXXX XXXX XXXX XXXX, tendering payment. Additionally, on XX/XX/year>, I mailed a XXXX XXXX XXXX Int

##### Finding — Clusters Show Both Category Alignment and Cross-Category Patterns

The clustering results show that some groups are strongly aligned with existing CFPB issue categories, while others contain complaints from several different categories.

For example, clusters 1, 3, and 8 are relatively concentrated around purchase disputes, investigation/report problems, and fees or interest respectively. Other clusters are considerably more mixed.

This suggests that the embeddings are capturing some existing complaint structure while also identifying relationships that cross the CFPB taxonomy.

However, the clusters are still broad and exploratory. They cannot yet be treated as customer problems without further analysis of the themes within them.

#### 8. Identifying Distinctive Cluster Language

The cluster examples show that some groups are more coherent than others.

I will now identify the terms that distinguish each cluster from the overall complaint dataset. This should provide a more systematic basis for understanding what each cluster represents.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

cluster_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_features=3000
)

cluster_tfidf = cluster_vectorizer.fit_transform(
    text_df["complaint_what_happened"]
)

features = cluster_vectorizer.get_feature_names_out()

for cluster in sorted(text_df["cluster"].unique()):
    cluster_rows = text_df["cluster"].values == cluster
    
    mean_scores = cluster_tfidf[cluster_rows].mean(axis=0).A1
    top_indices = mean_scores.argsort()[::-1][:10]
    
    print(f"\nCluster {cluster}:")
    print([
        (features[i], round(mean_scores[i], 4))
        for i in top_indices
    ])


Cluster 0:
[('xxxx', np.float64(0.1602)), ('xxxx xxxx', np.float64(0.1077)), ('credit', np.float64(0.0857)), ('report', np.float64(0.062)), ('consumer', np.float64(0.0617)), ('15', np.float64(0.0475)), ('credit report', np.float64(0.0468)), ('reporting', np.float64(0.0452)), ('xx', np.float64(0.0427)), ('information', np.float64(0.0424))]

Cluster 1:
[('xxxx', np.float64(0.223)), ('xxxx xxxx', np.float64(0.1097)), ('xx', np.float64(0.1071)), ('dispute', np.float64(0.0753)), ('xx xx', np.float64(0.0537)), ('refund', np.float64(0.0526)), ('merchant', np.float64(0.0433)), ('00', np.float64(0.0428)), ('xx xxxx', np.float64(0.0406)), ('charge', np.float64(0.0304))]

Cluster 2:
[('chase', np.float64(0.2615)), ('xxxx', np.float64(0.1836)), ('xxxx xxxx', np.float64(0.0927)), ('xx', np.float64(0.089)), ('card', np.float64(0.0627)), ('credit', np.float64(0.0575)), ('xx xx', np.float64(0.0445)), ('chase bank', np.float64(0.0428)), ('credit card', np.float64(0.0412)), ('account', np.float64(0.039

##### Finding — Cluster Quality Varies

The cluster-specific language shows that some clusters have recognizable themes, such as disputes, billing errors, promotional offers, fees, and fraud.

However, other clusters are dominated by anonymization tokens, company names, or general credit-card terms. This means the clustering does not produce equally meaningful customer themes across all groups.

The clusters will therefore be treated as exploratory signals rather than final product problems. The next step is to identify and validate the strongest themes using the actual complaint narratives.

#### 9. Validating Candidate Themes

Several clusters contain recognizable customer language, but cluster labels alone are not enough to establish a customer problem.

I will review representative complaints from the strongest candidate clusters to check whether the language and customer experiences are reasonably consistent.

In [ ]:
candidate_clusters = [1, 3, 5, 8, 9]

for cluster in candidate_clusters:
    cluster_data = text_df[text_df["cluster"] == cluster]

    print("\n" + "=" * 90)
    print("CLUSTER", cluster)

    for _, row in cluster_data.sample(
        min(5, len(cluster_data)),
        random_state=42
    ).iterrows():
        print("\nIssue:", row["issue"])
        print("Sub-issue:", row["sub_issue"])
        print("Narrative:", row["complaint_what_happened"][:700])


CLUSTER 1

Issue: Other features, terms, or problems
Sub-issue: Problem with rewards from credit card
Narrative: Name : XXXX XXXX Address : XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXXXX/XX/XXXXXXXX To whom it XXXX concern, Account number : XXXX ( they didn't give me the account number on the statement ) Dispute amount : {$25.00} Specific Transaction : {$26.00} purchased on XX/XX/year>. XXXX perfume. XXXX order number # XXXX Reason : I made a purchase of a perfume that I wanted to try out for the first time on XX/XX/year>. The transaction total is {$26.00} because I used {$100.00} from my XXXX XXXX XXXX and {$25.00} off from the CREDIT CARD REWARDS. 
I was allergic to the perfume, so I had to return it. XXXX customer service told me to call the credit card company once my perfume is processed through the re

Issue: Getting a credit card
Sub-issue: Card opened without my consent or knowledge
Narrative: They violated several rights from the consumer law while heisting my information and putt

##### Finding — Candidate Customer Themes

The cluster review identified several recurring themes that appear reasonably coherent across the customer narratives.

Examples include billing and late-payment errors, promotional or reward issues, fees and unexpected interest, and fraudulent or unauthorized charges.

Some themes also appear across different CFPB categories, suggesting that the AI analysis can provide a customer-experience view that differs from the existing taxonomy.

These themes are still candidates rather than final product problems. They need to be quantified and validated before prioritization.

#### 10. Quantifying Candidate Themes

The cluster review identified several potentially meaningful customer themes.

I will quantify the size of each cluster to understand which themes affect a substantial number of complaints. Cluster size will be used as one input for prioritization, not as the sole basis for selecting a product problem.


In [ ]:
cluster_sizes = (
    text_df["cluster"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="complaints")
)

cluster_sizes["percentage"] = (
    cluster_sizes["complaints"] / len(text_df) * 100
).round(2)

cluster_sizes

,cluster,complaints,percentage
0,0,793,7.93
1,1,1222,12.23
2,2,522,5.22
3,3,785,7.85
4,4,670,6.70
5,5,786,7.86
6,6,1473,14.74
7,7,1253,12.54
8,8,1243,12.44
9,9,1248,12.49


In [ ]:
candidate_theme_map = {
    1: "Purchase disputes / refunds",
    3: "Late payments / billing errors",
    5: "Promotions / rewards",
    8: "Fees / interest",
    9: "Fraud / unauthorized charges"
}

candidate_themes = cluster_sizes[
    cluster_sizes["cluster"].isin(candidate_theme_map)
].copy()

candidate_themes["candidate_theme"] = (
    candidate_themes["cluster"].map(candidate_theme_map)
)

candidate_themes[
    ["cluster", "candidate_theme", "complaints", "percentage"]
]

,cluster,candidate_theme,complaints,percentage
1,1,Purchase disputes / refunds,1222,12.23
3,3,Late payments / billing errors,785,7.85
5,5,Promotions / rewards,786,7.86
8,8,Fees / interest,1243,12.44
9,9,Fraud / unauthorized charges,1248,12.49


##### Finding — Candidate Theme Scale

The candidate clusters vary in size, with four of the five selected themes representing roughly 12% of the usable narratives and the late-payment/billing-error cluster representing about 8%.

Cluster size shows that these themes occur frequently enough to warrant further investigation, but it is not sufficient to determine product priority.

Some clusters also contain more than one type of customer experience, so the themes need to be refined before they can be treated as product opportunities.

#### 11. Refining Customer Themes

The initial clusters identify broad areas of customer concern, but broad topics are not yet product problems.

I will refine the strongest themes by looking at the recurring customer experience within each cluster and identifying the specific point of friction.

In [ ]:
focus_clusters = [3, 8, 9]

for cluster in focus_clusters:
    cluster_data = text_df[text_df["cluster"] == cluster]

    print("\n" + "=" * 90)
    print(f"CLUSTER {cluster} — {len(cluster_data)} complaints")

    for _, row in cluster_data.sample(8, random_state=42).iterrows():
        print(
            f"\nSub-issue: {row['sub_issue']}\n"
            f"{row['complaint_what_happened'][:800]}"
        )


CLUSTER 3 — 785 complaints

Sub-issue: Was not notified of investigation status or results
I've always ensured that payments on this account are made on time, never allowing them to become overdue. I'm unsure why there are reports of late payment remarks on my accounts. According to 15 USC 1666b - Billing Error, this should be rectified.

Sub-issue: Information belongs to someone else
This late payment is distressing because Ive consistently paid on time. I urge you to fix this error by removing it from my account. 
XXXX XXXX XXXXXXXX, US DEPT ED, US DEPT ED, XXXX XXXX, US DEPT ED, US DEPT ED, DPEDXXXX

Sub-issue: Was not notified of investigation status or results
I've consistently made timely payments on this account and have never allowed them to be overdue. I'm puzzled by the reports of late payment remarks on my accounts. According to 15 USC 1666b, billing errors should be corrected.

Sub-issue: Was not notified of investigation status or results
I never miss a payment. I am cert

##### Finding — From Themes to Customer Problems

Reviewing the narratives shows that the strongest themes describe more than a topic; they contain a recurring point of customer friction.

For example, the fraud-related cluster includes customers who reported unauthorized transactions, provided information or documentation, and still experienced rejected or unresolved disputes.

This is more actionable than simply labeling the theme as "fraud" because it identifies a specific customer experience that could potentially be addressed through product improvements.

The same approach will be used to refine the other candidate themes before prioritization.

#### 12. Candidate Customer Problems

The AI analysis identified several recurring themes, and reviewing the underlying narratives helped clarify the customer friction within them.

I will translate these themes into specific customer problems without selecting a priority yet.

| Theme | Candidate customer problem |
|---|---|
| Purchase disputes | Customers struggle to get disputed transactions resolved even after providing supporting information. |
| Late payments / billing errors | Customers report incorrect late-payment or billing information despite believing they paid correctly. |
| Promotions / rewards | Customers experience problems receiving advertised promotional benefits or rewards. |
| Fees / interest | Customers struggle to understand or resolve unexpected fees and interest charges. |
| Fraud / unauthorized charges | Customers struggle to resolve unauthorized transactions even after reporting them and providing supporting evidence. |

##### 13. Problem Validation

The candidate problems were identified from recurring themes in the AI-generated clusters.

Before prioritizing them, I will check how consistently the complaints within each candidate cluster fall into the same CFPB sub-issues.

A more concentrated cluster provides stronger evidence that the underlying customer experience is consistent, while a highly mixed cluster should be treated more cautiously.

In [ ]:
validation = []

for cluster in candidate_clusters:
    cluster_data = text_df[text_df["cluster"] == cluster]
    sub_issue_counts = cluster_data["sub_issue"].value_counts()

    top_sub_issue_share = (
        sub_issue_counts.iloc[0] / len(cluster_data) * 100
    )

    validation.append({
        "cluster": cluster,
        "candidate_theme": candidate_theme_map[cluster],
        "complaints": len(cluster_data),
        "top_sub_issue": sub_issue_counts.index[0],
        "top_sub_issue_share": round(top_sub_issue_share, 2)
    })

validation_df = pd.DataFrame(validation)

validation_df

,cluster,candidate_theme,complaints,top_sub_issue,top_sub_issue_share
0,1,Purchase disputes / refunds,1222,Credit card company isn't resolving a dispute about a purchase on your statement,70.95
1,3,Late payments / billing errors,785,Their investigation did not fix an error on your report,37.83
2,5,Promotions / rewards,786,Didn't receive advertised or promotional terms,25.06
3,8,Fees / interest,1243,Problem with fees,40.71
4,9,Fraud / unauthorized charges,1248,Credit card company isn't resolving a dispute about a purchase on your statement,36.30
